# Tabas - Base Price Modeling for Short-Term Rentals

Este notebook segue o case **Base Price Modeling for Short-Term Rentals** da Tabas.

Objetivo: estimar um **preco-base diario estrutural** para cada imovel usando dados de mercado do Airbnb, sem transformar o problema em precificacao dinamica. Por isso, datas de estadia e de consulta sao usadas para limpeza/agregacao e diagnostico, mas **nao entram como features finais de previsao**.

Principios adotados:

- robustez a outliers: filtros por quantis, mediana por anuncio e modelo em `log1p(preco)`;
- interpretabilidade: modelo linear regularizado com efeitos de bairro e atributos estruturais;
- raciocinio estatistico: validacao holdout por anuncio, metricas em reais e percentuais, analise por segmentos;
- producao: funcoes explicitas de limpeza, preparo de features, treino e inferencia.

## 1. Enquadramento do problema

O preco observado no Airbnb mistura valor estrutural do imovel com fatores dinamicos: sazonalidade, eventos, feriados, antecedencia, descontos e otimizacao por ocupacao. Como o case pede o **base nightly price**, a decisao central e reduzir o dado transacional de precos para um alvo mais estavel por `id`.

Decisoes principais:

1. Usar `price_per_night` como variavel-alvo de origem, pois ja esta no nivel diario.
2. Priorizar `los = 15`, que domina a base e tende a reduzir ruido de estadias curtas. `los = 3` fica para diagnostico e extensao futura.
3. Agregar multiplas cotacoes do mesmo imovel pela **mediana**, nao pela media, para reduzir impacto de datas/eventos e valores extremos.
4. Nao usar `check_in`, `check_out`, `reference_date` ou `extraction_date` como preditores do modelo estrutural.
5. Modelar `log1p(base_price)`, pois precos sao positivos, assimetricos e com cauda longa.

### Por que essas configurações?

> **`PRIMARY_LOS = 15`** – estadias de 15 noites dominam a base (~90 %) e diluem tarifas de curta duração, tornando o sinal de preço estrutural mais estável.  
> **`ENCODING = 'latin1'`** – necessário para preservar acentuação em colunas de bairros extraídas do Airbnb Brasil.  
> **`RANDOM_SEED = 42`** – garante reprodutibilidade em splits e qualquer embaralhamento futuro.


In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

DATA_DIR = Path("data/")
APART_PATH = DATA_DIR / "airbnb_apart.csv"
PRICES_PATH = DATA_DIR / "airbnb_prices.csv"

RANDOM_SEED = 42
PRIMARY_LOS = 15
ENCODING = "latin1" 

print(APART_PATH.exists(), PRICES_PATH.exists())

## 2. Funcoes auxiliares

O CSV de imoveis pode conter linhas de texto malformadas. A leitura abaixo usa `engine="python"` e `on_bad_lines="skip"` para manter a analise reprodutivel. Em producao, eu registraria essas linhas em uma tabela de qualidade para retorno ao pipeline de scraping.

### Por que essas funções auxiliares?

- **`read_apartments`**: garante tipagem correta e mantém apenas a observação estrutural mais recente por anúncio (`keep='last'` após sort por data de extração).  
- **`read_prices`**: carrega apenas as colunas necessárias, evitando pressão de memória com 1 M+ de linhas.  
- **Métricas customizadas** (MAE, RMSE, MAPE, MdAPE, R²): implementadas sem `sklearn` para manter o ambiente mínimo e facilitar auditoria.


In [ ]:
def read_apartments(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding=ENCODING, engine="python", on_bad_lines="skip")
    df = df.rename(columns={"Unnamed: 0": "source_row"})
    df["id"] = df["id"].astype(str)
    df["extraction_date"] = pd.to_datetime(df["extraction_date"], errors="coerce")

    for col in ["guests", "bedrooms", "beds", "bathrooms", "lat", "lon"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in ["is_superhost", "is_new_listing"]:
        df[col] = (
            df[col].astype(str).str.strip().str.lower()
            .map({"true": True, "false": False, "1": True, "0": False})
        )

    # Mantem a observacao estrutural mais recente por anuncio.
    df = (
        df.sort_values(["id", "extraction_date"])
        .drop_duplicates("id", keep="last")
        .reset_index(drop=True)
    )
    return df


def read_prices(path: Path) -> pd.DataFrame:
    usecols = [
        "id", "review_count", "check_in", "check_out", "los", "reference_date",
        "price_per_night", "communication_rating", "guest_satisfaction_rating",
        "accuracy_rating", "value_rating", "location_rating", "cleanliness_rating",
        "extraction_date",
    ]
    df = pd.read_csv(path, encoding=ENCODING, usecols=usecols)
    df["id"] = df["id"].astype(str)
    for col in ["check_in", "check_out", "reference_date", "extraction_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    numeric = [c for c in df.columns if c not in ["id", "check_in", "check_out", "reference_date", "extraction_date"]]
    for col in numeric:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def rmse(y, pred):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(pred)) ** 2)))


def mae(y, pred):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(pred))))


def mape(y, pred):
    y = np.asarray(y)
    pred = np.asarray(pred)
    mask = y > 0
    return float(np.mean(np.abs((y[mask] - pred[mask]) / y[mask])))


def mdape(y, pred):
    y = np.asarray(y)
    pred = np.asarray(pred)
    mask = y > 0
    return float(np.median(np.abs((y[mask] - pred[mask]) / y[mask])))


def r2_score(y, pred):
    y = np.asarray(y)
    pred = np.asarray(pred)
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return float(1 - ss_res / ss_tot)

### O que esperamos encontrar?

Antes de modelar, auditamos o dado bruto para detectar inconsistências (IDs duplicados, coordenadas inválidas, preços absurdos). A correspondência entre os dois arquivos (`matched_ids`) indica o tamanho real do universo modelável.


## 3. Carregamento e auditoria inicial

In [ ]:
apart = read_apartments(APART_PATH)
prices = read_prices(PRICES_PATH)

summary = {
    "apart_rows_after_read": len(apart),
    "apart_unique_ids": apart["id"].nunique(),
    "price_rows": len(prices),
    "price_unique_ids": prices["id"].nunique(),
    "matched_ids": len(set(apart["id"]) & set(prices["id"])),
}
summary

In [ ]:
display(apart.head())
display(prices.head())
display(apart[["guests", "bedrooms", "beds", "bathrooms", "lat", "lon"]].describe(percentiles=[.01, .05, .5, .95, .99]).T)
display(prices[["price_per_night", "los", "review_count", "guest_satisfaction_rating"]].describe(percentiles=[.01, .05, .5, .95, .99]).T)
display(prices["los"].value_counts(dropna=False).to_frame("rows").head(20))

### Por que transformação logarítmica?

Preços de imóveis seguem distribuição log-normal: cauda direita longa, assimetria positiva forte. Modelar `log1p(preço)` serve a dois propósitos:
1. Aproxima a distribuição da normal, melhorando o ajuste da regressão;
2. Faz os coeficientes legíveis em escala multiplicativa (percentual).


## 4. EDA: distribuicao de preco e outliers

A distribuicao de `price_per_night` costuma ser assimetrica. Valores extremos podem vir de eventos, erros de scraping, configuracoes de disponibilidade ou estadias muito especificas. Para o modelo-base, o foco e a faixa recorrente de mercado, nao a cauda extrema.

In [ ]:
price_audit = prices["price_per_night"].describe(percentiles=[.001, .01, .05, .25, .5, .75, .95, .99, .999]).to_frame("price_per_night")
display(price_audit)

los_price = (
    prices.query("price_per_night > 0")
    .groupby("los")["price_per_night"]
    .agg(rows="size", listings="nunique", median="median", mean="mean", p95=lambda s: s.quantile(.95), p99=lambda s: s.quantile(.99))
    .sort_values("rows", ascending=False)
)
display(los_price)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

valid_prices = prices.loc[prices['price_per_night'].gt(0), 'price_per_night']
cap = valid_prices.quantile(0.99)
clipped = valid_prices.clip(upper=cap)
log_vals = np.log1p(valid_prices)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Distribuição de preço por noite (cap p99)',
                    'Distribuição de log1p(preço)'])

fig.add_trace(go.Histogram(x=clipped, nbinsx=60, name='Preço R$',
    marker_color='#636EFA', opacity=0.8), row=1, col=1)
fig.add_trace(go.Histogram(x=log_vals, nbinsx=60, name='log1p(preço)',
    marker_color='#EF553B', opacity=0.8), row=1, col=2)

fig.update_layout(height=400, width=1000, showlegend=False,
    title_text='EDA – Distribuição de Preço', template='plotly_white')
fig.update_xaxes(title_text='R$ por noite', row=1, col=1)
fig.update_xaxes(title_text='log1p(R$ por noite)', row=1, col=2)
fig.update_yaxes(title_text='Cotações', row=1, col=1)
fig.show()


### Decisão de agregação

Cada anúncio pode aparecer em dezenas de cotações (datas, janelas de antecedência). Usar a **mediana** por `id` em vez da média:
- é robusta a cotações de pico (eventos, feriados);
- preserva o sinal estrutural sem requerer remoção agressiva de outliers.

O filtro `[p0.5%, p99.5%]` em nível de cotação elimina apenas erros grosseiros de scraping (preços = 0 ou valores > R$ 300k/noite).


## 5. Construcao do alvo estrutural

Escolha: usar `los = 15` como janela primaria.

Justificativa:

- e a duracao mais frequente na base;
- estadias de 15 noites tendem a diluir taxas/ruidos de estadias curtas;
- usar datas diretamente faria o modelo aprender sazonalidade/eventos, que o case explicitamente exclui;
- a mediana por `id` reduz o impacto de cotacoes ocasionais muito altas/baixas.

In [ ]:
def build_listing_target(prices: pd.DataFrame, primary_los: int = PRIMARY_LOS) -> pd.DataFrame:
    p = prices.copy()
    p = p[p["price_per_night"].notna() & (p["price_per_night"] > 0)]
    p = p[p["los"].eq(primary_los)].copy()

    # Filtro leve em nivel de cotacao para remover erros/cauda extrema sem apagar heterogeneidade real.
    lo, hi = p["price_per_night"].quantile([0.005, 0.995])
    p = p[p["price_per_night"].between(lo, hi)]

    agg = (
        p.groupby("id")
        .agg(
            base_price=("price_per_night", "median"),
            price_mean=("price_per_night", "mean"),
            price_p25=("price_per_night", lambda s: s.quantile(.25)),
            price_p75=("price_per_night", lambda s: s.quantile(.75)),
            n_price_quotes=("price_per_night", "size"),
            n_reference_dates=("reference_date", "nunique"),
            first_check_in=("check_in", "min"),
            last_check_in=("check_in", "max"),
            review_count=("review_count", "median"),
            communication_rating=("communication_rating", "median"),
            guest_satisfaction_rating=("guest_satisfaction_rating", "median"),
            accuracy_rating=("accuracy_rating", "median"),
            value_rating=("value_rating", "median"),
            location_rating=("location_rating", "median"),
            cleanliness_rating=("cleanliness_rating", "median"),
        )
        .reset_index()
    )
    agg["target_iqr"] = agg["price_p75"] - agg["price_p25"]
    agg["target_cv_proxy"] = agg["target_iqr"] / agg["base_price"].replace(0, np.nan)
    return agg


target = build_listing_target(prices)
display(target.describe(percentiles=[.01, .05, .5, .95, .99]).T)
target.head()

In [ ]:
model_df = apart.merge(target, on="id", how="inner")

# Limpeza estrutural conservadora. Valores impossiveis viram NaN para imputacao, nao exclusao ampla.
for col in ["guests", "bedrooms", "beds", "bathrooms"]:
    model_df.loc[model_df[col] < 0, col] = np.nan

model_df.loc[model_df["bedrooms"] > 10, "bedrooms"] = np.nan
model_df.loc[model_df["beds"] > 20, "beds"] = np.nan
model_df.loc[model_df["bathrooms"] > 10, "bathrooms"] = np.nan
model_df.loc[model_df["guests"] > 20, "guests"] = np.nan

# Coordenadas fora da regiao provavel de Sao Paulo sao tratadas como ausentes.
model_df.loc[~model_df["lat"].between(-25, -22), "lat"] = np.nan
model_df.loc[~model_df["lon"].between(-48, -45), "lon"] = np.nan

# Outliers residuais no alvo agregado.
target_lo, target_hi = model_df["base_price"].quantile([0.01, 0.99])
model_df = model_df[model_df["base_price"].between(target_lo, target_hi)].copy()

model_df["log_base_price"] = np.log1p(model_df["base_price"])
model_df["beds_per_bedroom"] = model_df["beds"] / model_df["bedrooms"].replace(0, np.nan)
model_df["guests_per_bedroom"] = model_df["guests"] / model_df["bedrooms"].replace(0, np.nan)
model_df["bathrooms_per_bedroom"] = model_df["bathrooms"] / model_df["bedrooms"].replace(0, np.nan)
model_df["log_review_count"] = np.log1p(model_df["review_count"])

print(model_df.shape)
display(model_df[["base_price", "guests", "bedrooms", "beds", "bathrooms", "lat", "lon", "n_price_quotes"]].describe(percentiles=[.01, .05, .5, .95, .99]).T)

### O que os dados de bairro revelam?

A tabela de bairros serve como **sanidade check** antes de modelar: bairros como Bertioga e Riviera apresentam medianas 5-8× maiores que bairros periféricos, justificando o uso de dummies de bairro como features.


## 6. EDA: bairro e atributos estruturais

In [ ]:
neighborhood_summary = (
    model_df.groupby("airbnb_neighborhood")
    .agg(
        listings=("id", "nunique"),
        median_base_price=("base_price", "median"),
        p25=("base_price", lambda s: s.quantile(.25)),
        p75=("base_price", lambda s: s.quantile(.75)),
        median_bedrooms=("bedrooms", "median"),
    )
    .query("listings >= 20")
    .sort_values("median_base_price", ascending=False)
)
display(neighborhood_summary.head(20))
display(neighborhood_summary.tail(20))

size_summary = (
    model_df.groupby("bedrooms")
    .agg(listings=("id", "nunique"), median_base_price=("base_price", "median"), p25=("base_price", lambda s: s.quantile(.25)), p75=("base_price", lambda s: s.quantile(.75)))
    .sort_index()
)
display(size_summary)

In [ ]:
import plotly.express as px

top_neigh = neighborhood_summary.head(20).copy()
top_neigh = top_neigh.sort_values('median_base_price')

fig = px.bar(
    top_neigh.reset_index(),
    x='median_base_price', y='airbnb_neighborhood',
    orientation='h',
    text='listings',
    color='median_base_price',
    color_continuous_scale='Blues',
    title='Top 20 bairros por preço-base mediano (mín. 20 anúncios)',
    labels={'median_base_price': 'R$ por noite', 'airbnb_neighborhood': 'Bairro',
            'listings': 'Anúncios'},
    template='plotly_white', height=600
)
fig.update_traces(textposition='outside')
fig.show()

# Preço mediano por número de quartos
fig2 = px.box(
    model_df[model_df['bedrooms'].between(1,5)],
    x='bedrooms', y='base_price',
    color='bedrooms',
    title='Distribuição de preço-base por nº de quartos',
    labels={'bedrooms': 'Quartos', 'base_price': 'Preço-base (R$)'},
    template='plotly_white', height=450
)
fig2.update_layout(showlegend=False)
fig2.show()


### Escolha do modelo: Ridge em log-preço

**Por que não Gradient Boosting ou Random Forest?**
- O case prioriza **interpretabilidade**: coeficientes ridge têm leitura direta em escala log → percentual.  
- Regularização L2 é natural para dados com muitas categorias de bairro (one-hot encoding → matriz esparsa).  
- Mais fácil de versionar, auditar e reimplementar em produção (apenas um vetor de pesos).  

> Em etapa posterior, Gradient Boosting seria comparado como *benchmark* de performance.


## 7. Modelo

Modelo escolhido: **regressao ridge em log-preco** com one-hot encoding para categorias.

Por que nao um modelo mais complexo como boosting?

- o case prioriza interpretabilidade e raciocinio;
- uma regressao regularizada e simples de operacionalizar, debugar e monitorar;
- efeitos de bairro e atributos estruturais ficam diretamente interpretaveis;
- `log1p(preco)` faz os coeficientes terem leitura aproximada em percentual.

Em uma etapa posterior, eu compararia contra Gradient Boosting/Random Forest como benchmark de performance, mantendo este modelo como baseline interpretavel.

In [ ]:
NUMERIC_FEATURES = [
    "guests", "bedrooms", "beds", "bathrooms", "lat", "lon",
    "beds_per_bedroom", "guests_per_bedroom", "bathrooms_per_bedroom",
    "log_review_count", "guest_satisfaction_rating", "accuracy_rating",
    "value_rating", "location_rating", "cleanliness_rating",
    "n_price_quotes", "target_cv_proxy",
]

CATEGORICAL_FEATURES = [
    "realty_type", "airbnb_neighborhood", "listing_obj_type",
    "is_superhost", "is_new_listing", "city",
]


def train_test_split_by_id(df: pd.DataFrame, test_size=.2, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    ids = df["id"].drop_duplicates().to_numpy()
    rng.shuffle(ids)
    n_test = int(round(len(ids) * test_size))
    test_ids = set(ids[:n_test])
    is_test = df["id"].isin(test_ids)
    return df.loc[~is_test].copy(), df.loc[is_test].copy()


def fit_preprocessor(train: pd.DataFrame, numeric_features, categorical_features, min_category_count=25):
    prep = {
        "numeric_features": list(numeric_features),
        "categorical_features": list(categorical_features),
        "numeric_medians": {},
        "numeric_means": {},
        "numeric_stds": {},
        "categories": {},
    }

    for col in numeric_features:
        x = pd.to_numeric(train[col], errors="coerce")
        med = float(x.median()) if x.notna().any() else 0.0
        filled = x.fillna(med)
        mean = float(filled.mean())
        std = float(filled.std(ddof=0))
        prep["numeric_medians"][col] = med
        prep["numeric_means"][col] = mean
        prep["numeric_stds"][col] = std if std > 1e-12 else 1.0

    for col in categorical_features:
        s = train[col].astype("object").where(train[col].notna(), "MISSING").astype(str)
        counts = s.value_counts()
        cats = counts[counts >= min_category_count].index.tolist()
        prep["categories"][col] = cats

    return prep


def transform_features(df: pd.DataFrame, prep):
    parts = []
    names = []

    for col in prep["numeric_features"]:
        x = pd.to_numeric(df[col], errors="coerce").fillna(prep["numeric_medians"][col])
        z = (x - prep["numeric_means"][col]) / prep["numeric_stds"][col]
        parts.append(z.to_numpy().reshape(-1, 1))
        names.append(col)

    for col in prep["categorical_features"]:
        s = df[col].astype("object").where(df[col].notna(), "MISSING").astype(str)
        cats = prep["categories"][col]
        for cat in cats:
            parts.append((s == cat).astype(float).to_numpy().reshape(-1, 1))
            names.append(f"{col}={cat}")

    X = np.hstack(parts) if parts else np.empty((len(df), 0))
    intercept = np.ones((len(df), 1))
    return np.hstack([intercept, X]), ["intercept"] + names


def fit_ridge_closed_form(X, y, alpha=10.0, sample_weight=None):
    y = np.asarray(y, dtype=float)
    if sample_weight is None:
        w = np.ones(len(y))
    else:
        w = np.asarray(sample_weight, dtype=float)
    sw = np.sqrt(w)
    Xw = X * sw[:, None]
    yw = y * sw

    penalty = np.eye(X.shape[1]) * alpha
    penalty[0, 0] = 0.0  # intercepto nao regularizado
    beta = np.linalg.solve(Xw.T @ Xw + penalty, Xw.T @ yw)
    return beta


def predict(beta, X):
    return np.expm1(X @ beta)

In [ ]:
train_df, test_df = train_test_split_by_id(model_df)
prep = fit_preprocessor(train_df, NUMERIC_FEATURES, CATEGORICAL_FEATURES)

X_train, feature_names = transform_features(train_df, prep)
X_test, _ = transform_features(test_df, prep)

y_train_log = train_df["log_base_price"].to_numpy()
y_test = test_df["base_price"].to_numpy()

# Mais cotacoes por anuncio aumentam confianca no alvo, mas com raiz para nao dominar o treino.
weights = np.sqrt(train_df["n_price_quotes"].clip(lower=1).to_numpy())

candidate_alphas = [0.1, 1, 3, 10, 30, 100, 300]
rows = []
for alpha in candidate_alphas:
    beta = fit_ridge_closed_form(X_train, y_train_log, alpha=alpha, sample_weight=weights)
    pred_test = predict(beta, X_test)
    rows.append({
        "alpha": alpha,
        "MAE": mae(y_test, pred_test),
        "RMSE": rmse(y_test, pred_test),
        "MAPE": mape(y_test, pred_test),
        "MdAPE": mdape(y_test, pred_test),
        "R2": r2_score(y_test, pred_test),
    })

validation_table = pd.DataFrame(rows).sort_values("MAE")
display(validation_table)

best_alpha = float(validation_table.iloc[0]["alpha"])
beta = fit_ridge_closed_form(X_train, y_train_log, alpha=best_alpha, sample_weight=weights)
test_df = test_df.copy()
test_df["prediction"] = predict(beta, X_test)
print("best_alpha:", best_alpha)

### Como interpretar as métricas?

| Métrica | O que mede | Por que usar |
|---------|-----------|-------------|
| **MAE** | Erro médio absoluto em R\$ | Comunicável para pricing/operações |
| **RMSE** | Penaliza erros grandes | Exposição em imóveis caros |
| **MAPE** | Erro percentual médio | R\$ 80 pesa diferente em R\$ 200 vs R\$ 900 |
| **MdAPE** | Mediana do erro percentual | Robusto a outliers de previsão |
| **R²** | Poder explicativo geral | Diagnóstico secundário |


## 8. Avaliacao

In [ ]:
overall_metrics = {
    "MAE_R$": mae(test_df["base_price"], test_df["prediction"]),
    "RMSE_R$": rmse(test_df["base_price"], test_df["prediction"]),
    "MAPE": mape(test_df["base_price"], test_df["prediction"]),
    "MdAPE": mdape(test_df["base_price"], test_df["prediction"]),
    "R2": r2_score(test_df["base_price"], test_df["prediction"]),
    "test_listings": len(test_df),
}
overall_metrics

In [ ]:
def segment_metrics(df, segment_col, min_n=30):
    out = []
    for value, g in df.groupby(segment_col, dropna=False):
        if len(g) < min_n:
            continue
        out.append({
            "segment": value,
            "n": len(g),
            "actual_median": g["base_price"].median(),
            "pred_median": g["prediction"].median(),
            "MAE_R$": mae(g["base_price"], g["prediction"]),
            "MAPE": mape(g["base_price"], g["prediction"]),
            "bias_R$": float((g["prediction"] - g["base_price"]).mean()),
        })
    return pd.DataFrame(out).sort_values("MAPE")


test_df["price_quartile"] = pd.qcut(test_df["base_price"], 4, labels=["Q1 barato", "Q2", "Q3", "Q4 caro"])
test_df["bedroom_bucket"] = test_df["bedrooms"].clip(upper=4).fillna(-1).astype(int).astype(str)

display(segment_metrics(test_df, "price_quartile", min_n=1))
display(segment_metrics(test_df, "bedroom_bucket", min_n=20))
display(segment_metrics(test_df, "airbnb_neighborhood", min_n=20).sort_values("MAPE").head(20))
display(segment_metrics(test_df, "airbnb_neighborhood", min_n=20).sort_values("MAPE", ascending=False).head(20))

### Análise visual das métricas por segmento

Metricas usadas:

- **MAE**: erro medio absoluto em reais; facil de comunicar para pricing e operacoes.
- **RMSE**: penaliza erros grandes; util para verificar exposicao em imoveis caros.
- **MAPE/MdAPE**: erro percentual; importante porque um erro de R$ 80 pesa diferente em um studio de R$ 200 e em um apartamento de R$ 900.
- **R2**: poder explicativo geral, usado como diagnostico secundario.

### Interpretando coeficientes log-lineares

Em `log1p(preço) = Xβ`, o impacto de uma feature numérica é:
```
Δpreço% ≈ exp(β_padronizado / σ_feature) - 1
```
Para dummies de bairro, o coeficiente mede o deslocamento percentual *relativo à categoria de referência implícita*, controlando pelos demais atributos.


## 9. Interpretacao dos drivers

In [ ]:
coef = pd.DataFrame({"feature": feature_names, "coef_log": beta})
coef["abs_coef"] = coef["coef_log"].abs()
display(coef.query("feature != 'intercept'").sort_values("abs_coef", ascending=False).head(40))


def numeric_effect(feature):
    # Impacto aproximado de +1 unidade em feature numerica no preco esperado.
    idx = feature_names.index(feature)
    std = prep["numeric_stds"][feature]
    coef_per_unit = beta[idx] / std
    pct = math.exp(coef_per_unit) - 1
    return {"feature": feature, "coef_log_per_unit": coef_per_unit, "approx_pct_impact": pct}


effects = pd.DataFrame([
    numeric_effect("bedrooms"),
    numeric_effect("bathrooms"),
    numeric_effect("guests"),
    numeric_effect("beds"),
])
display(effects)

### Visualização dos coeficientes

In [ ]:
import plotly.express as px

# Top 30 features by absolute coefficient
_top = coef.query("feature != 'intercept'").nlargest(30, 'abs_coef').copy()
_top['cor'] = _top['coef_log'].apply(lambda v: 'Positivo' if v > 0 else 'Negativo')
fig = px.bar(
    _top.sort_values('coef_log'),
    x='coef_log', y='feature', orientation='h',
    color='cor', color_discrete_map={'Positivo':'#2ecc71','Negativo':'#e74c3c'},
    title='Top 30 drivers do modelo (coeficiente em log-preço)',
    labels={'coef_log':'Coeficiente log', 'feature':'Feature'},
    template='plotly_white', height=750
)
fig.show()


In [ ]:
neigh_effects = coef[coef["feature"].str.startswith("airbnb_neighborhood=")].copy()
neigh_effects["neighborhood"] = neigh_effects["feature"].str.replace("airbnb_neighborhood=", "", regex=False)
neigh_effects["approx_pct_vs_reference"] = np.exp(neigh_effects["coef_log"]) - 1
neigh_counts = train_df["airbnb_neighborhood"].value_counts().rename("train_listings")
neigh_effects = neigh_effects.merge(neigh_counts, left_on="neighborhood", right_index=True, how="left")

display(neigh_effects.sort_values("coef_log", ascending=False).head(20)[["neighborhood", "train_listings", "approx_pct_vs_reference", "coef_log"]])
display(neigh_effects.sort_values("coef_log").head(20)[["neighborhood", "train_listings", "approx_pct_vs_reference", "coef_log"]])

Como ler os coeficientes:

- Em um modelo log-linear, coeficientes numericos podem ser convertidos em impacto percentual aproximado por `exp(coeficiente_por_unidade) - 1`.
- Para bairros, o efeito e relativo a categoria de referencia implicita e condicionado aos demais atributos. Portanto, a leitura correta e "dado um imovel estruturalmente parecido, estar neste bairro desloca o preco esperado em X% contra a referencia".
- Coeficientes muito altos em bairros com poucas observacoes devem ser tratados com cautela. Por isso, categorias raras sao agrupadas e a regularizacao ridge reduz volatilidade.

### Função de inferência para produção

A função `predict_base_price` encapsula todo o pipeline (imputação, padronização, encoding, predição) em uma única chamada, recebendo um DataFrame no mesmo formato do CSV de aptos. Isso facilita integração em pipelines de batch scoring e APIs REST.


## 10. Funcao de inferencia para producao

In [ ]:
model_artifact = {
    "best_alpha": best_alpha,
    "feature_names": feature_names,
    "beta": beta.tolist(),
    "preprocessor": prep,
    "primary_los": PRIMARY_LOS,
    "target_definition": "median price_per_night by listing using los=15 after quote-level p0.5/p99.5 filtering",
}


def predict_base_price(new_listings: pd.DataFrame, artifact=model_artifact) -> pd.Series:
    df = new_listings.copy()
    for col in ["guests", "bedrooms", "beds", "bathrooms", "lat", "lon", "review_count", "n_price_quotes", "target_cv_proxy"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["beds_per_bedroom"] = df["beds"] / df["bedrooms"].replace(0, np.nan)
    df["guests_per_bedroom"] = df["guests"] / df["bedrooms"].replace(0, np.nan)
    df["bathrooms_per_bedroom"] = df["bathrooms"] / df["bedrooms"].replace(0, np.nan)
    if "log_review_count" not in df.columns:
        df["log_review_count"] = np.log1p(pd.to_numeric(df.get("review_count", 0), errors="coerce").fillna(0))
    if "n_price_quotes" not in df.columns:
        df["n_price_quotes"] = 1
    if "target_cv_proxy" not in df.columns:
        df["target_cv_proxy"] = np.nan

    X_new, _ = transform_features(df, artifact["preprocessor"])
    pred = predict(np.asarray(artifact["beta"]), X_new)
    return pd.Series(pred, index=new_listings.index, name="predicted_base_price")


sample_predictions = test_df.head(10).copy()
sample_predictions["production_prediction"] = predict_base_price(sample_predictions)
display(sample_predictions[["id", "airbnb_neighborhood", "bedrooms", "bathrooms", "base_price", "prediction", "production_prediction"]])

### Próximos passos prioritários

1. **Precificação dinâmica**: modelar multiplicadores sobre o preço-base (sazonalidade, eventos, antecedência).  
2. **Dados textuais**: embeddings de `ad_title` e `kicker_content_message` podem capturar atributos de luxo não estruturados.  
3. **Hierarquia geográfica**: modelo de efeitos mistos (bairro → cidade) para shrinkage automático em bairros com poucas amostras.  
4. **Monitoramento**: drift de MAPE por bairro e faixa de preço a cada ciclo de reextração.


## 11. Limitacoes e proximos passos

Limitacoes:

- O alvo ainda e derivado de cotacoes Airbnb, entao pode conter residuos de sazonalidade e descontos.
- `los = 15` aumenta consistencia, mas pode representar um mercado de estadias medias, nao necessariamente diaria curta.
- Bairros com pouca amostra precisam de shrinkage/agrupamento adicional.
- Ratings podem ser endogenos: precos altos influenciam percepcao de valor e vice-versa.
- Dados textuais foram preservados, mas nao usados no modelo final para manter simplicidade e interpretabilidade.

Como operacionalizar na Tabas:

1. Reprocessar precos diariamente/semanalmente e atualizar o alvo mediano por anuncio.
2. Treinar por cidade ou com efeitos hierarquicos de cidade quando houver multiplas cidades.
3. Monitorar MAE/MAPE por bairro, tipologia e faixa de preco.
4. Versionar artefato do modelo, pre-processador e definicao do alvo.
5. Usar o preco-base como ancora; aplicar camada separada para sazonalidade, eventos, ocupacao e antecedencia.

Bonus: extensao para precificacao dinamica

- Depois de consolidar o preco-base, modelar multiplicadores dinamicos sobre a ancora estrutural.
- Features dinamicas possiveis: dia da semana, feriados, eventos, antecedencia, ocupacao interna, ritmo de reservas, elasticidade por bairro e concorrencia ativa.
- A arquitetura recomendada e multiplicativa: `preco_final = preco_base * fator_sazonal * fator_evento * fator_ocupacao * fator_antecedencia`, com limites e regras de negocio.

In [ ]:
# Export opcional do artefato para uso posterior.
artifact_path = Path("models/tabas_base_price_model_artifact.json")
with artifact_path.open("w", encoding="utf-8") as f:
    json.dump(model_artifact, f, ensure_ascii=False, indent=2)
print(f"Artefato salvo em: {artifact_path.resolve()}")